# Chapter 2: Data Structures & Algorithms
## Notebook 03 - Advanced

Trees, graphs, and dynamic programming: the structures behind neural networks, knowledge graphs, and optimization.

**What you'll learn:**
- Binary trees and traversals
- Graphs: BFS and DFS
- Dynamic programming fundamentals
- Capstone: building a simple text search index

**Time estimate:** 1.5 hours

---
*Generated by Berta AI | Created by Luigi Giacobbe*

## 1. Trees

Trees are hierarchical structures. In AI:
- Decision trees are a core ML algorithm
- Parse trees represent sentence structure in NLP
- File systems and JSON data are tree-structured

In [ ]:
class TreeNode:
    """A binary tree node."""
    def __init__(self, value, left=None, right=None):
        self.value = value
        self.left = left
        self.right = right


def inorder(node):
    """Left -> Root -> Right (gives sorted order for BST)."""
    if node is None:
        return []
    return inorder(node.left) + [node.value] + inorder(node.right)


def preorder(node):
    """Root -> Left -> Right (used to serialize trees)."""
    if node is None:
        return []
    return [node.value] + preorder(node.left) + preorder(node.right)


def tree_height(node):
    """Height of a tree (depth of deepest leaf)."""
    if node is None:
        return 0
    return 1 + max(tree_height(node.left), tree_height(node.right))


# Build a simple decision tree structure
#        accuracy > 0.9?
#       /              \
#    deploy          retrain?
#                   /       \
#              more_data   change_model

tree = TreeNode("accuracy > 0.9?",
    left=TreeNode("deploy"),
    right=TreeNode("retrain?",
        left=TreeNode("more_data"),
        right=TreeNode("change_model")
    )
)

print(f"In-order:  {inorder(tree)}")
print(f"Pre-order: {preorder(tree)}")
print(f"Height:    {tree_height(tree)}")

## 2. Graphs

Graphs model relationships: social networks, knowledge graphs, neural network architectures, dependency graphs.

In [ ]:
from collections import deque


class Graph:
    """A directed graph using adjacency list."""
    
    def __init__(self):
        self.edges = {}  # node -> [neighbors]
    
    def add_edge(self, src, dst):
        self.edges.setdefault(src, []).append(dst)
        self.edges.setdefault(dst, [])  # ensure dst exists
    
    def bfs(self, start):
        """Breadth-first search: explore level by level."""
        visited = set()
        queue = deque([start])
        order = []
        
        while queue:
            node = queue.popleft()
            if node not in visited:
                visited.add(node)
                order.append(node)
                for neighbor in self.edges.get(node, []):
                    if neighbor not in visited:
                        queue.append(neighbor)
        return order
    
    def dfs(self, start, visited=None):
        """Depth-first search: explore as deep as possible first."""
        if visited is None:
            visited = set()
        
        visited.add(start)
        order = [start]
        
        for neighbor in self.edges.get(start, []):
            if neighbor not in visited:
                order.extend(self.dfs(neighbor, visited))
        return order
    
    def has_path(self, src, dst):
        """Check if there's a path from src to dst."""
        return dst in self.bfs(src)


# Model a simple ML pipeline dependency graph
pipeline = Graph()
pipeline.add_edge("raw_data", "clean_data")
pipeline.add_edge("clean_data", "features")
pipeline.add_edge("clean_data", "labels")
pipeline.add_edge("features", "train_model")
pipeline.add_edge("labels", "train_model")
pipeline.add_edge("train_model", "evaluate")
pipeline.add_edge("evaluate", "deploy")

print("Pipeline BFS (execution order):")
for step in pipeline.bfs("raw_data"):
    print(f"  -> {step}")

print(f"\nPath raw_data -> deploy? {pipeline.has_path('raw_data', 'deploy')}")
print(f"Path deploy -> raw_data? {pipeline.has_path('deploy', 'raw_data')}")

## 3. Dynamic Programming

DP solves complex problems by breaking them into overlapping subproblems. In AI, DP appears in sequence alignment (NLP), Viterbi algorithm (HMMs), and optimal policy computation (RL).

In [ ]:
def edit_distance(s1, s2):
    """Levenshtein distance: min edits to transform s1 into s2.
    
    Used in NLP for: spell checking, fuzzy matching, sequence alignment.
    """
    m, n = len(s1), len(s2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    
    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j
    
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if s1[i-1] == s2[j-1]:
                dp[i][j] = dp[i-1][j-1]
            else:
                dp[i][j] = 1 + min(
                    dp[i-1][j],    # deletion
                    dp[i][j-1],    # insertion
                    dp[i-1][j-1],  # substitution
                )
    
    return dp[m][n]


# Spell checking: find closest word
vocabulary = ["machine", "learning", "deep", "neural", "network",
              "training", "dataset", "model", "algorithm", "tensor"]

misspelled = "machin"
distances = [(word, edit_distance(misspelled, word)) for word in vocabulary]
distances.sort(key=lambda x: x[1])

print(f"Spell check for '{misspelled}':")
for word, dist in distances[:5]:
    print(f"  {word:>12}: {dist} edit(s)")

# Similarity between model names
pairs = [("gpt-4", "gpt-3"), ("bert", "bart"), ("llama", "alpaca")]
for a, b in pairs:
    print(f"\n  '{a}' vs '{b}': {edit_distance(a, b)} edits")

## 4. Capstone: Simple Text Search Index

Combining hash tables, sorting, and searching to build a text search index - the foundation of information retrieval systems used in RAG.

In [ ]:
from collections import defaultdict
import math


class SearchIndex:
    """A simple inverted index for text search (TF-IDF based)."""
    
    def __init__(self):
        self.index = defaultdict(list)  # word -> [(doc_id, frequency)]
        self.documents = {}             # doc_id -> text
        self.doc_lengths = {}           # doc_id -> word count
    
    def add_document(self, doc_id, text):
        """Index a document."""
        self.documents[doc_id] = text
        words = text.lower().split()
        self.doc_lengths[doc_id] = len(words)
        
        word_freq = defaultdict(int)
        for word in words:
            clean = word.strip(".,!?;:'\"")
            if clean:
                word_freq[clean] += 1
        
        for word, freq in word_freq.items():
            self.index[word].append((doc_id, freq))
    
    def search(self, query, top_k=5):
        """Search documents using TF-IDF scoring."""
        query_words = query.lower().split()
        scores = defaultdict(float)
        n_docs = len(self.documents)
        
        for word in query_words:
            if word not in self.index:
                continue
            
            doc_freq = len(self.index[word])
            idf = math.log(n_docs / (1 + doc_freq))
            
            for doc_id, freq in self.index[word]:
                tf = freq / max(self.doc_lengths[doc_id], 1)
                scores[doc_id] += tf * idf
        
        ranked = sorted(scores.items(), key=lambda x: -x[1])
        return ranked[:top_k]
    
    def stats(self):
        return {
            "documents": len(self.documents),
            "unique_terms": len(self.index),
            "avg_doc_length": sum(self.doc_lengths.values()) / max(len(self.doc_lengths), 1),
        }


# Build a mini search engine for AI topics
index = SearchIndex()

docs = {
    "d1": "Neural networks learn patterns from training data using backpropagation",
    "d2": "Transformers use attention mechanisms for natural language processing",
    "d3": "Reinforcement learning trains agents through rewards and exploration",
    "d4": "RAG systems retrieve relevant documents to augment language model generation",
    "d5": "Fine-tuning adapts pre-trained models to specific domains and tasks",
    "d6": "Convolutional neural networks excel at image recognition and processing",
    "d7": "Large language models generate text using transformer architectures",
    "d8": "Data preprocessing is essential for training accurate machine learning models",
}

for doc_id, text in docs.items():
    index.add_document(doc_id, text)

print(f"Index stats: {index.stats()}")

queries = ["neural network training", "language models", "reinforcement rewards"]
for query in queries:
    results = index.search(query, top_k=3)
    print(f"\nQuery: '{query}'")
    for doc_id, score in results:
        print(f"  [{score:.4f}] {doc_id}: {docs[doc_id][:60]}...")

## Chapter 2 Complete!

You've mastered the fundamental data structures and algorithms used in AI:

- **Notebook 01**: Arrays, Big O, linear/binary search, two-pointer, sliding window
- **Notebook 02**: Stacks, queues, hash tables, merge sort, quicksort
- **Notebook 03**: Trees, graphs, BFS/DFS, dynamic programming, search index

Next up: **Chapter 3: Linear Algebra & Calculus** - the math that powers neural networks.

---
*Generated by Berta AI | Created by Luigi Giacobbe*